In [1]:
!pip install spacy pandas geopy textblob
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.9 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Read txt
with open("orwell.txt", "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# Find the London part
start = text.find("XXIV")
text_london = text[start:]

text_london = text_london.replace("\n", " ")

# Process London part
doc = nlp(text_london)

In [3]:
# DisplaCy visualizer (named entities)
from spacy import displacy
displacy.render(doc, style="ent", jupyter=True)

In [4]:
# SpaCy extract places
places = []
 
for ent in doc.ents:
    if ent.label_ in ["GPE", "LOC", "FAC"]:
        places.append({
            "place": ent.text,
            "sentence": ent.sent.text.strip(),
            "method": "spacy"
        })

# Regex supplement
import re

street_matches = re.findall(
    r"\b[A-Z][a-z]+\s(?:Street|Road|Avenue|Lane|Square|Hill|Bridge|Market)\b",
    text_london
)

for s in street_matches:
    for sent in doc.sents:
        if s in sent.text:
            places.append({
                "place": s,
                "sentence": sent.text.strip(),
                "method": "regex"
            })
            break

# Remove duplicates
unique_places = {}

for p in places:
    key = p["place"].strip()
    if key not in unique_places:
        unique_places[key] = p

places_clean = list(unique_places.values())

In [5]:
# Transform to real-world coordinates
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="orwell_project")

results = []

for p in places_clean:
    try:
        location = geolocator.geocode(p["place"] + ", London")

        if location:
            address = location.raw.get("display_name", "")

            if "London" in address and "United Kingdom" in address:
                results.append({
                    "place": p["place"],
                    "lat": location.latitude,
                    "lon": location.longitude,
                    "context": p["sentence"]
                })

        time.sleep(1)

    except:
        continue

In [6]:
# Export and save as a CSV file
import pandas as pd

df = pd.DataFrame(results)

df = df[["place", "lat", "lon", "context"]]
df = df.drop_duplicates(subset="place").reset_index(drop=True)

df.to_csv("places.csv", index=False, encoding="utf-8-sig")

In [7]:
import re

# Sentences -> sent1,sent2,sent3...
sentences = list(doc.sents)   

results_money = []

currency_units = [
    "pound", "pounds", "£",
    "shilling", "shillings",
    "penny", "pence",
    "crown", "crowns"
]

for idx, sent in enumerate(sentences):
    sentence_text = sent.text.strip()

    money_phrases = []

    # spaCy money
    money_phrases += [ent.text for ent in sent.ents if ent.label_ == "MONEY"]

    # number + units
    tokens = list(sent)
    for i, token in enumerate(tokens[:-1]):
        if token.like_num:
            next_token = tokens[i+1].text.lower()
            if next_token in currency_units:
                money_phrases.append(token.text + " " + tokens[i+1].text)

    # Currency symbol
    matches = re.findall(r"£\s?\d+", sentence_text)
    money_phrases += matches

    # Supplementary recognition and deduplication
    matches = re.findall(r"\b(a|one|half)(?:\s+a)?\s+(penny|crown|shilling)\b", sentence_text, re.IGNORECASE)
    money_phrases += [" ".join(m) for m in matches]

    money_phrases = list(dict.fromkeys(money_phrases))

    if money_phrases:
        # Context
        context_sentences = []

        if idx > 0:
            context_sentences.append(sentences[idx-1].text.strip())

        context_sentences.append(sentence_text)

        if idx < len(sentences) - 1:
            context_sentences.append(sentences[idx+1].text.strip())

        results_money.append({
            "money": money_phrases,
            "context": " ".join(context_sentences) 
        })

In [8]:
# Export and save as a CSV file
df_money = pd.DataFrame(results_money)
df_money.head()
df_money.to_csv("money.csv", index=False, encoding="utf-8-sig")